# Brain Tumor MRI Classification — VGG16 Transfer Learning
### Full Retraining Notebook

**Dataset:** 12,064 T1-weighted contrast-enhanced MRI images  
**Classes:** Glioma · Meningioma · Pituitary · No Tumor  
**Architecture:** VGG16 (ImageNet) + custom classification head  
**Strategy:** Two-phase training (head → fine-tune top 30 VGG layers)

---

## Notebook Flow
| Section | Description |
|---------|-------------|
| 1 | Environment & path verification |
| 2 | Data loading & augmentation |
| 3 | Class distribution & weights |
| 4 | Build model (Functional API) |
| 5 | Phase 1 — Train classification head |
| 6 | Phase 2 — Fine-tune top VGG16 blocks |
| 7 | Evaluation (classification report, confusion matrix, AUC) |
| 8 | Grad-CAM visualisation |
| 9 | Save final model |

## 1 — Environment Setup

In [ ]:
import os
import sys
import warnings
warnings.filterwarnings('ignore')

# ── Ensure project root is on the path ──────────────────────────────────────
# The notebook lives in  <project_root>/notebook/
# All src.* imports resolve from project root one level up.
NOTEBOOK_DIR = os.path.abspath('.')
PROJECT_ROOT = os.path.dirname(NOTEBOOK_DIR)   # go up from notebook/
if PROJECT_ROOT not in sys.path:
    sys.path.insert(0, PROJECT_ROOT)

print(f"Project root : {PROJECT_ROOT}")
print(f"Notebook dir : {NOTEBOOK_DIR}")

# Quick sanity — confirm src package is importable
try:
    import src
    print("✅ src package found")
except ImportError as e:
    print(f"❌ src package not found: {e}")
    print("   Make sure this notebook is inside <project_root>/notebook/")

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
import seaborn as sns
import tensorflow as tf
from tensorflow.keras.applications.vgg16 import preprocess_input, VGG16
from tensorflow.keras import layers, models
from tensorflow.keras.optimizers import Adam
from tensorflow.keras.callbacks import EarlyStopping, ModelCheckpoint, ReduceLROnPlateau
from tensorflow.keras.preprocessing.image import ImageDataGenerator
from tensorflow.keras.preprocessing import image as keras_image
from tensorflow.keras.models import load_model
from sklearn.utils.class_weight import compute_class_weight
from sklearn.metrics import (
    classification_report, confusion_matrix,
    roc_auc_score, roc_curve, auc
)
from sklearn.preprocessing import label_binarize
from PIL import Image
import time

print(f"TensorFlow  : {tf.__version__}")
print(f"GPUs found  : {tf.config.list_physical_devices('GPU')}")
print(f"NumPy       : {np.__version__}")

## 2 — Configuration

All paths and hyperparameters in one place. Adjust `PROJECT_ROOT` if you run the notebook from a different location.

In [ ]:
# ── Paths ────────────────────────────────────────────────────────────────────
TRAIN_DIR  = os.path.join(PROJECT_ROOT, 'data', 'train')
TEST_DIR   = os.path.join(PROJECT_ROOT, 'data', 'test')
MODEL_DIR  = os.path.join(PROJECT_ROOT, 'models')
PLOTS_DIR  = os.path.join(PROJECT_ROOT, 'outputs', 'plots')
REPORTS_DIR= os.path.join(PROJECT_ROOT, 'outputs', 'reports')

# The model will be saved here — matches config.py
BEST_MODEL_PATH  = os.path.join(MODEL_DIR, 'brain_tumor_vgg16.keras')
FINAL_MODEL_PATH = os.path.join(MODEL_DIR, 'final_model.keras')

for d in [MODEL_DIR, PLOTS_DIR, REPORTS_DIR]:
    os.makedirs(d, exist_ok=True)

# ── Hyperparameters ──────────────────────────────────────────────────────────
IMG_SIZE         = (224, 224)
BATCH_SIZE       = 32
RANDOM_SEED      = 42

# Must match folder names exactly (case-sensitive)
CLASSES          = ['Glioma', 'Meningioma', 'Pituitary', 'No_Tumor']
NUM_CLASSES      = len(CLASSES)

PHASE1_EPOCHS    = 20      # head training (EarlyStopping will cut short if needed)
PHASE2_EPOCHS    = 20      # fine-tuning
LEARNING_RATE    = 1e-4    # Phase 1 LR
FINE_TUNE_LR     = 1e-5    # Phase 2 LR (10x smaller to preserve VGG features)
PATIENCE         = 5       # EarlyStopping patience
UNFREEZE_FROM    = -30     # unfreeze last 30 VGG16 layers in Phase 2

tf.random.set_seed(RANDOM_SEED)
np.random.seed(RANDOM_SEED)

# ── Path verification ────────────────────────────────────────────────────────
print("Path check:")
for label, path in [("Train dir", TRAIN_DIR), ("Test dir", TEST_DIR), ("Model dir", MODEL_DIR)]:
    exists = os.path.isdir(path)
    status = "✅" if exists else "❌ NOT FOUND"
    print(f"  {label:<12} {status}  →  {path}")

print("\nClass folder check:")
for cls in CLASSES:
    tr = os.path.isdir(os.path.join(TRAIN_DIR, cls))
    te = os.path.isdir(os.path.join(TEST_DIR,  cls))
    print(f"  {cls:<14} train={'✅' if tr else '❌'}   test={'✅' if te else '❌'}")

## 3 — Dataset Statistics & Class Distribution

In [ ]:
def count_images(base_dir, classes):
    counts = {}
    for cls in classes:
        cls_dir = os.path.join(base_dir, cls)
        if os.path.isdir(cls_dir):
            n = len([f for f in os.listdir(cls_dir)
                     if f.lower().endswith(('.jpg', '.jpeg', '.png'))])
        else:
            n = 0
        counts[cls] = n
    return counts

train_counts = count_images(TRAIN_DIR, CLASSES)
test_counts  = count_images(TEST_DIR,  CLASSES)

print(f"{'Class':<16} {'Train':>8} {'Test':>8} {'Total':>8}")
print("-" * 44)
for cls in CLASSES:
    total = train_counts[cls] + test_counts[cls]
    print(f"  {cls:<14} {train_counts[cls]:>8,} {test_counts[cls]:>8,} {total:>8,}")
print("-" * 44)
tr_total = sum(train_counts.values())
te_total = sum(test_counts.values())
print(f"  {'TOTAL':<14} {tr_total:>8,} {te_total:>8,} {tr_total+te_total:>8,}")

In [ ]:
# Bar chart — class distribution
fig, axes = plt.subplots(1, 2, figsize=(12, 4))

x = np.arange(NUM_CLASSES)
colors = ['#E74C3C', '#3498DB', '#2ECC71', '#9B59B6']

for ax, counts, title in zip(axes,
                              [train_counts, test_counts],
                              ['Training Set', 'Test Set']):
    vals = [counts[c] for c in CLASSES]
    bars = ax.bar(CLASSES, vals, color=colors, edgecolor='white', linewidth=1.2)
    ax.set_title(title, fontsize=13, fontweight='bold')
    ax.set_ylabel('Image Count')
    ax.set_ylim(0, max(vals) * 1.2)
    for bar, v in zip(bars, vals):
        ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 30,
                f'{v:,}', ha='center', fontsize=10, fontweight='bold')
    ax.tick_params(axis='x', labelsize=10)

fig.suptitle('Dataset Class Distribution', fontsize=14, fontweight='bold', y=1.02)
fig.tight_layout()
fig.savefig(os.path.join(PLOTS_DIR, 'class_distribution.png'), dpi=150, bbox_inches='tight')
plt.show()
plt.close(fig)
print("Saved → class_distribution.png")

## 4 — Data Generators & Augmentation

**Augmentation applied to training only:**
- Random rotation ±20°
- Width / height shift ±20%
- Shear ±20°
- Zoom ±20%
- Horizontal flip

VGG16 channel-wise mean subtraction (`preprocess_input`) is applied to all splits.

In [ ]:
# ── Training + validation generators ────────────────────────────────────────
train_datagen = ImageDataGenerator(
    preprocessing_function=preprocess_input,
    rotation_range=20,
    width_shift_range=0.2,
    height_shift_range=0.2,
    shear_range=0.2,
    zoom_range=0.2,
    horizontal_flip=True,
    validation_split=0.2,
)

common_args = dict(
    directory=TRAIN_DIR,
    target_size=IMG_SIZE,
    batch_size=BATCH_SIZE,
    class_mode='categorical',
    classes=CLASSES,        # forces deterministic label order
    seed=RANDOM_SEED,
)

train_gen = train_datagen.flow_from_directory(subset='training',   shuffle=True,  **common_args)
val_gen   = train_datagen.flow_from_directory(subset='validation', shuffle=False, **common_args)

# ── Test generator (no augmentation) ────────────────────────────────────────
test_datagen = ImageDataGenerator(preprocessing_function=preprocess_input)
test_gen = test_datagen.flow_from_directory(
    TEST_DIR,
    target_size=IMG_SIZE,
    batch_size=BATCH_SIZE,
    class_mode='categorical',
    classes=CLASSES,
    shuffle=False,
)

print(f"\nLabel map : {train_gen.class_indices}")
print(f"Train batches : {len(train_gen)}")
print(f"Val   batches : {len(val_gen)}")
print(f"Test  batches : {len(test_gen)}")

In [ ]:
# ── Visualise sample augmented images ────────────────────────────────────────
mean_bgr = np.array([103.939, 116.779, 123.68])   # VGG16 channel means

def undo_preprocess(img):
    """Approximate un-preprocessing for display only."""
    return np.clip((img + mean_bgr) / 255.0, 0, 1)

batch_imgs, batch_lbls = next(train_gen)
fig, axes = plt.subplots(2, 4, figsize=(14, 7))
fig.suptitle('Sample Augmented Training Images', fontsize=13, fontweight='bold')

for ax, img, lbl in zip(axes.flat, batch_imgs[:8], batch_lbls[:8]):
    ax.imshow(undo_preprocess(img))
    ax.set_title(CLASSES[np.argmax(lbl)], fontsize=10)
    ax.axis('off')

fig.tight_layout()
fig.savefig(os.path.join(PLOTS_DIR, 'sample_augmented.png'), dpi=150, bbox_inches='tight')
plt.show()
plt.close(fig)

## 5 — Class Weights

The dataset is imbalanced (Glioma has ~55% more images than No_Tumor in training).  
`compute_class_weight('balanced')` assigns higher loss weight to minority classes so the model does not simply learn to predict the majority class.

In [ ]:
y_train_labels  = train_gen.classes
unique_classes  = np.unique(y_train_labels)
weights         = compute_class_weight('balanced', classes=unique_classes, y=y_train_labels)
class_weight_dict = dict(zip(unique_classes.tolist(), weights.tolist()))

print("Class weights (higher = minority class gets more attention during training):")
for idx, w in class_weight_dict.items():
    print(f"  [{idx}] {CLASSES[idx]:<14}  weight = {w:.4f}")

## 6 — Model Architecture

**Why Functional API instead of Sequential?**

`Sequential([VGG16(...), ...])` wraps the base model as an opaque block — Keras cannot access individual layer tensors. This breaks:
- Grad-CAM (needs `layer.output` tensors)
- Fine-tuning layer inspection
- `model.inputs` / `model.outputs`

The Functional API (`Model(inputs, outputs)`) keeps the entire graph transparent.

**Architecture summary:**
```
Input (224×224×3)
    ↓
VGG16 backbone (13 conv layers, 5 max-pool) — frozen in Phase 1
    ↓  (7×7×512)
GlobalAveragePooling2D
    ↓  (512,)
Dense(256, ReLU)
    ↓
Dropout(0.5)
    ↓
Dense(4, Softmax)
```

In [ ]:
def build_model(freeze_base: bool = True):
    """Build VGG16 transfer-learning model using the Functional API."""
    base = VGG16(weights='imagenet', include_top=False, input_shape=(224, 224, 3))
    base.trainable = not freeze_base

    inputs  = base.input
    x       = base.output
    x       = layers.GlobalAveragePooling2D()(x)
    x       = layers.Dense(256, activation='relu')(x)
    x       = layers.Dropout(0.5)(x)
    outputs = layers.Dense(NUM_CLASSES, activation='softmax')(x)

    model = models.Model(inputs=inputs, outputs=outputs, name='brain_tumor_vgg16')
    return model, base

model, base_model = build_model(freeze_base=True)
model.summary()

# Confirm graph tensors are exposed (required for Grad-CAM)
print(f"\nModel inputs  : {model.inputs}")
print(f"Model outputs : {model.outputs}")
print(f"\nTotal params        : {model.count_params():,}")
print(f"Trainable params    : {sum(tf.size(w).numpy() for w in model.trainable_weights):,}")
print(f"Non-trainable params: {sum(tf.size(w).numpy() for w in model.non_trainable_weights):,}")

## 7 — Callbacks

In [ ]:
def make_callbacks(phase_name: str):
    """
    Return a fresh set of callbacks for each training phase.
    Fresh instances avoid state bleeding between Phase 1 and Phase 2
    (e.g. ModelCheckpoint's best-score tracker resets cleanly).
    """
    return [
        EarlyStopping(
            monitor='val_accuracy',
            patience=PATIENCE,
            restore_best_weights=True,
            verbose=1,
        ),
        ModelCheckpoint(
            BEST_MODEL_PATH,
            save_best_only=True,
            monitor='val_accuracy',
            mode='max',
            verbose=1,
        ),
        ReduceLROnPlateau(
            monitor='val_loss',
            factor=0.2,
            patience=3,
            min_lr=1e-7,
            verbose=1,
        ),
    ]

def plot_history(history, title: str, suffix: str = ''):
    """Plot accuracy and loss curves side by side."""
    fig, axes = plt.subplots(1, 2, figsize=(13, 4))
    fig.suptitle(title, fontsize=14, fontweight='bold')

    for ax, metric, label in zip(axes, ['accuracy', 'loss'], ['Accuracy', 'Loss']):
        ax.plot(history.history[metric],     label='Train', linewidth=2)
        ax.plot(history.history[f'val_{metric}'], label='Val',   linewidth=2, linestyle='--')
        ax.set_title(label, fontsize=12)
        ax.set_xlabel('Epoch')
        ax.legend()
        ax.grid(True, alpha=0.3)

    fig.tight_layout()
    path = os.path.join(PLOTS_DIR, f'history{suffix}.png')
    fig.savefig(path, dpi=150, bbox_inches='tight')
    plt.show()
    plt.close(fig)
    print(f"Saved → {path}")

print("Callbacks and plotting helpers ready.")

## 8 — Phase 1: Train Classification Head

VGG16 base is **frozen** — only the 3 new layers (GAP → Dense → Dropout → Dense) are trained.

- LR: `1e-4`
- Epochs: up to 20 (EarlyStopping with patience=5)
- Class weights applied to handle imbalance

In [ ]:
model.compile(
    optimizer=Adam(learning_rate=LEARNING_RATE),
    loss='categorical_crossentropy',
    metrics=['accuracy'],
)

print("─── Phase 1: Training classification head ───")
print(f"Trainable layers: {len([l for l in model.layers if l.trainable])}")
print(f"Learning rate   : {LEARNING_RATE}")
print(f"Max epochs      : {PHASE1_EPOCHS}\n")

t0 = time.time()
history1 = model.fit(
    train_gen,
    epochs=PHASE1_EPOCHS,
    validation_data=val_gen,
    class_weight=class_weight_dict,
    callbacks=make_callbacks('phase1'),
)
phase1_time = time.time() - t0
print(f"\n⏱  Phase 1 completed in {phase1_time/60:.1f} minutes")

In [ ]:
plot_history(history1, title='Phase 1 — Classification Head Training', suffix='_phase1')

# Print best metrics
best_epoch = np.argmax(history1.history['val_accuracy'])
print(f"Best epoch        : {best_epoch + 1}")
print(f"Best val accuracy : {history1.history['val_accuracy'][best_epoch]:.4f}")
print(f"Best val loss     : {history1.history['val_loss'][best_epoch]:.4f}")

## 9 — Phase 2: Fine-tune Top VGG16 Layers

Unfreeze the **last 30 layers** of the VGG16 base. Earlier layers remain frozen because they capture universal low-level features (edges, textures) that transfer well to any image domain.

The last 30 layers (block4_conv2 onwards) capture higher-level patterns that can adapt to the specific textures and structures of brain MRI.

- LR: `1e-5` (10× smaller than Phase 1 to avoid destroying pre-trained weights)
- Must recompile after changing `trainable` flags (Keras requirement)

In [ ]:
# Unfreeze last 30 layers of VGG16 base
base_model.trainable = True
for layer in base_model.layers[:UNFREEZE_FROM]:
    layer.trainable = False

trainable_now = [l.name for l in base_model.layers if l.trainable]
print(f"Unfrozen VGG16 layers ({len(trainable_now)}):")
for name in trainable_now:
    print(f"  • {name}")

# Must recompile after changing trainable flags
model.compile(
    optimizer=Adam(learning_rate=FINE_TUNE_LR),
    loss='categorical_crossentropy',
    metrics=['accuracy'],
)

print(f"\nLearning rate : {FINE_TUNE_LR}")
print(f"Max epochs    : {PHASE2_EPOCHS}")

In [ ]:
print("─── Phase 2: Fine-tuning top VGG16 layers ───\n")
t0 = time.time()
history2 = model.fit(
    train_gen,
    epochs=PHASE2_EPOCHS,
    validation_data=val_gen,
    class_weight=class_weight_dict,
    callbacks=make_callbacks('phase2'),
)
phase2_time = time.time() - t0
print(f"\n⏱  Phase 2 completed in {phase2_time/60:.1f} minutes")

In [ ]:
plot_history(history2, title='Phase 2 — Fine-tuning', suffix='_phase2')

best_epoch2 = np.argmax(history2.history['val_accuracy'])
print(f"Best epoch        : {best_epoch2 + 1}")
print(f"Best val accuracy : {history2.history['val_accuracy'][best_epoch2]:.4f}")
print(f"Best val loss     : {history2.history['val_loss'][best_epoch2]:.4f}")

In [ ]:
# ── Combined training curve across both phases ────────────────────────────────
def combine_histories(h1, h2):
    combined = {}
    for key in h1.history:
        combined[key] = h1.history[key] + h2.history[key]
    return combined

combined = combine_histories(history1, history2)
phase1_end = len(history1.history['accuracy'])

fig, axes = plt.subplots(1, 2, figsize=(14, 5))
fig.suptitle('Full Training History (Phase 1 + Phase 2)', fontsize=14, fontweight='bold')

for ax, metric, label in zip(axes, ['accuracy', 'loss'], ['Accuracy', 'Loss']):
    ax.plot(combined[metric],         label='Train', linewidth=2, color='#2196F3')
    ax.plot(combined[f'val_{metric}'],label='Val',   linewidth=2, color='#FF9800', linestyle='--')
    ax.axvline(phase1_end - 0.5, color='red', linestyle=':', linewidth=1.5, alpha=0.7,
               label=f'Phase 2 starts (epoch {phase1_end})')
    ax.set_title(label, fontsize=12)
    ax.set_xlabel('Epoch')
    ax.legend()
    ax.grid(True, alpha=0.3)

fig.tight_layout()
path = os.path.join(PLOTS_DIR, 'history_combined.png')
fig.savefig(path, dpi=150, bbox_inches='tight')
plt.show()
plt.close(fig)
print(f"Saved → {path}")

## 10 — Evaluation on Test Set

Loading the **best checkpoint** (saved by `ModelCheckpoint` monitoring `val_accuracy`), then evaluating on the held-out test set.

In [ ]:
# Load best checkpoint
best_model = load_model(BEST_MODEL_PATH, compile=False)
print(f"✅ Best model loaded from {BEST_MODEL_PATH}")

# ── Load all test images manually ────────────────────────────────────────────
# We load manually (not via generator) to guarantee label order matches CLASSES.
class_to_idx = {cls: i for i, cls in enumerate(CLASSES)}
X_test, y_true = [], []

print("\nLoading test images …")
for cls in CLASSES:
    cls_dir = os.path.join(TEST_DIR, cls)
    loaded  = 0
    for fname in sorted(os.listdir(cls_dir)):
        if not fname.lower().endswith(('.jpg', '.jpeg', '.png')):
            continue
        img_path = os.path.join(cls_dir, fname)
        try:
            img = keras_image.load_img(img_path, target_size=IMG_SIZE)
            arr = preprocess_input(keras_image.img_to_array(img))
            X_test.append(arr)
            y_true.append(class_to_idx[cls])
            loaded += 1
        except Exception as e:
            print(f"  Skipping {fname}: {e}")
    print(f"  {cls:<14} {loaded:>4} images  (label={class_to_idx[cls]})")

X_test = np.array(X_test, dtype=np.float32)
y_true = np.array(y_true, dtype=np.int32)
print(f"\nTotal: {len(X_test):,} test images loaded")

In [ ]:
# ── Batch inference ───────────────────────────────────────────────────────────
BATCH = 32
parts = []
print(f"Running inference ({(len(X_test)+BATCH-1)//BATCH} batches) …")
for i in range(0, len(X_test), BATCH):
    preds = best_model.predict(X_test[i:i+BATCH], verbose=0)
    parts.append(preds)

y_pred_prob = np.concatenate(parts, axis=0)
y_pred      = np.argmax(y_pred_prob, axis=1)
print("Done.")

In [ ]:
# ── Classification report ──────────────────────────────────────────────────
report = classification_report(y_true, y_pred, target_names=CLASSES, digits=4)
print("Classification Report:")
print(report)

with open(os.path.join(REPORTS_DIR, 'classification_report.txt'), 'w') as f:
    f.write(report)
print(f"Saved → {os.path.join(REPORTS_DIR, 'classification_report.txt')}")

In [ ]:
# ── Confusion matrix ───────────────────────────────────────────────────────
cm  = confusion_matrix(y_true, y_pred)
cm_pct = cm.astype(float) / cm.sum(axis=1, keepdims=True) * 100

fig, axes = plt.subplots(1, 2, figsize=(15, 6))
fig.suptitle('Confusion Matrix — Test Set', fontsize=14, fontweight='bold')

# Raw counts
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues',
            xticklabels=CLASSES, yticklabels=CLASSES, ax=axes[0],
            linewidths=0.5, linecolor='white')
axes[0].set_title('Counts')
axes[0].set_ylabel('True Label')
axes[0].set_xlabel('Predicted Label')

# Row-normalised percentages
sns.heatmap(cm_pct, annot=True, fmt='.1f', cmap='Blues',
            xticklabels=CLASSES, yticklabels=CLASSES, ax=axes[1],
            linewidths=0.5, linecolor='white',
            vmin=0, vmax=100)
axes[1].set_title('Row-Normalised (%)')
axes[1].set_ylabel('True Label')
axes[1].set_xlabel('Predicted Label')

fig.tight_layout()
path = os.path.join(PLOTS_DIR, 'confusion_matrix.png')
fig.savefig(path, dpi=150, bbox_inches='tight')
plt.show()
plt.close(fig)
print(f"Saved → {path}")

In [ ]:
# ── Per-class AUC-ROC ──────────────────────────────────────────────────────
y_true_bin = label_binarize(y_true, classes=list(range(NUM_CLASSES)))
auc_scores = roc_auc_score(y_true_bin, y_pred_prob, average=None)

print("Per-class AUC-ROC:")
for cls, auc_val in zip(CLASSES, auc_scores):
    print(f"  {cls:<14} {auc_val:.4f}")
print(f"\n  Macro-average AUC : {np.mean(auc_scores):.4f}")

# ── ROC curves ───────────────────────────────────────────────────────────────
colors = ['#E74C3C', '#3498DB', '#2ECC71', '#9B59B6']
fig, ax = plt.subplots(figsize=(8, 6))

for i, (cls, color) in enumerate(zip(CLASSES, colors)):
    fpr, tpr, _ = roc_curve(y_true_bin[:, i], y_pred_prob[:, i])
    roc_auc = auc(fpr, tpr)
    ax.plot(fpr, tpr, color=color, linewidth=2,
            label=f'{cls}  (AUC = {roc_auc:.4f})')

ax.plot([0,1], [0,1], 'k--', linewidth=1, label='Random classifier')
ax.set_xlabel('False Positive Rate')
ax.set_ylabel('True Positive Rate')
ax.set_title('ROC Curves — Test Set', fontsize=13, fontweight='bold')
ax.legend(loc='lower right', fontsize=10)
ax.grid(True, alpha=0.3)
fig.tight_layout()
path = os.path.join(PLOTS_DIR, 'roc_curves.png')
fig.savefig(path, dpi=150, bbox_inches='tight')
plt.show()
plt.close(fig)
print(f"Saved → {path}")

## 11 — Grad-CAM Visualisation

**What is Grad-CAM?**  
Gradient-weighted Class Activation Mapping highlights which pixels in the input image most influenced the model's prediction. Red/yellow regions = high attention, blue = low attention.

**How it works:**
1. Build a sub-model that outputs both the last conv-layer activations and the final prediction.
2. Use `GradientTape` to compute the gradient of the predicted class score w.r.t. those activations.
3. Global-average-pool the gradients → per-channel importance weights.
4. Weighted sum of activation maps → ReLU → normalise → heatmap.

In [ ]:
import cv2
from src.gradcam import make_gradcam_heatmap, overlay_heatmap, find_last_conv_layer

# Confirm the target conv layer is found
last_conv = find_last_conv_layer(best_model)
print(f"Target conv layer : {last_conv.name}")
print(f"Output shape      : {last_conv.output_shape}")

In [ ]:
def show_gradcam_grid(test_dir, model, classes, n_per_class=2):
    """Show a grid of Original | Grad-CAM overlays for each class."""
    fig = plt.figure(figsize=(5 * n_per_class * 2, 5 * len(classes)))
    fig.suptitle('Grad-CAM — Model Attention Maps', fontsize=16, fontweight='bold', y=1.01)

    gs = gridspec.GridSpec(len(classes), n_per_class * 2,
                           hspace=0.4, wspace=0.15)

    for row, cls in enumerate(classes):
        cls_dir = os.path.join(test_dir, cls)
        img_files = sorted([
            f for f in os.listdir(cls_dir)
            if f.lower().endswith(('.jpg','.jpeg','.png'))
        ])[:n_per_class]

        for col, fname in enumerate(img_files):
            img_path = os.path.join(cls_dir, fname)

            # Load original
            pil_img  = Image.open(img_path).convert('RGB')
            orig_arr = np.array(pil_img)

            # Preprocess for model
            img_resized = pil_img.resize(IMG_SIZE, Image.LANCZOS)
            img_arr     = keras_image.img_to_array(img_resized)
            img_pre     = preprocess_input(img_arr.copy())

            # Predict
            probs    = best_model.predict(img_pre[np.newaxis], verbose=0)[0]
            pred_idx = int(np.argmax(probs))
            conf     = float(probs[pred_idx])

            # Grad-CAM
            try:
                heatmap = make_gradcam_heatmap(img_pre, best_model, pred_index=pred_idx)
                overlay = overlay_heatmap(heatmap,
                                          np.array(img_resized),
                                          alpha=0.45)
            except Exception as e:
                print(f"  Grad-CAM failed for {fname}: {e}")
                overlay = np.array(img_resized)

            correct = (classes[pred_idx] == cls)
            border  = '#27AE60' if correct else '#E74C3C'

            # Original subplot
            ax_orig = fig.add_subplot(gs[row, col * 2])
            ax_orig.imshow(img_resized)
            ax_orig.set_title(f'True: {cls}', fontsize=9)
            ax_orig.axis('off')
            for spine in ax_orig.spines.values():
                spine.set_visible(True)
                spine.set_color(border)
                spine.set_linewidth(3)

            # Grad-CAM subplot
            ax_cam = fig.add_subplot(gs[row, col * 2 + 1])
            ax_cam.imshow(overlay)
            tick = '✓' if correct else '✗'
            ax_cam.set_title(
                f'Pred: {classes[pred_idx]} {tick} ({conf:.1%})', fontsize=9
            )
            ax_cam.axis('off')

    path = os.path.join(PLOTS_DIR, 'gradcam_grid.png')
    fig.savefig(path, dpi=150, bbox_inches='tight')
    plt.show()
    plt.close(fig)
    print(f"Saved → {path}")

show_gradcam_grid(TEST_DIR, best_model, CLASSES, n_per_class=2)

## 12 — Single Image Prediction

Change `IMG_PATH` to any MRI image you want to test.

In [ ]:
from src.predict import predict_image

# Pick the first image from the first class as a demo
demo_dir  = os.path.join(TEST_DIR, CLASSES[0])
demo_file = sorted([f for f in os.listdir(demo_dir)
                    if f.lower().endswith(('.jpg','.jpeg','.png'))])[0]
IMG_PATH  = os.path.join(demo_dir, demo_file)

# ── Predict ──────────────────────────────────────────────────────────────────
predicted_class, confidence, all_probs = predict_image(best_model, IMG_PATH)

print(f"Image      : {os.path.basename(IMG_PATH)}")
print(f"Prediction : {predicted_class}  ({confidence:.2%} confidence)")
print("\nAll probabilities:")
for cls, p in zip(CLASSES, all_probs):
    bar = '█' * int(p * 30)
    print(f"  {cls:<14} {p:.4f}  {bar}")

In [ ]:
# ── Visual prediction card ────────────────────────────────────────────────────
pil_img = Image.open(IMG_PATH).convert('RGB')
img_resized = pil_img.resize(IMG_SIZE, Image.LANCZOS)
img_arr = keras_image.img_to_array(img_resized)
img_pre = preprocess_input(img_arr.copy())

heatmap = make_gradcam_heatmap(img_pre, best_model,
                                pred_index=int(np.argmax(all_probs)))
overlay = overlay_heatmap(heatmap, np.array(img_resized), alpha=0.45)

fig = plt.figure(figsize=(14, 4))
gs  = gridspec.GridSpec(1, 3, width_ratios=[1, 1, 1.4], wspace=0.3)

# Original
ax0 = fig.add_subplot(gs[0])
ax0.imshow(img_resized)
ax0.set_title('Original MRI', fontsize=11)
ax0.axis('off')

# Grad-CAM
ax1 = fig.add_subplot(gs[1])
ax1.imshow(overlay)
ax1.set_title(f'Grad-CAM ({predicted_class})', fontsize=11)
ax1.axis('off')

# Probabilities
ax2 = fig.add_subplot(gs[2])
colors = ['#E74C3C' if c == predicted_class else '#BDC3C7' for c in CLASSES]
bars = ax2.barh(CLASSES, all_probs, color=colors, edgecolor='white')
ax2.set_xlim(0, 1.1)
ax2.set_title('Class Probabilities', fontsize=11)
ax2.set_xlabel('Probability')
for bar, p in zip(bars, all_probs):
    ax2.text(p + 0.02, bar.get_y() + bar.get_height()/2,
             f'{p:.3f}', va='center', fontsize=10)
ax2.invert_yaxis()

fig.suptitle(
    f'Prediction: {predicted_class}  |  Confidence: {confidence:.2%}',
    fontsize=13, fontweight='bold'
)
fig.tight_layout()
path = os.path.join(PLOTS_DIR, 'prediction_card.png')
fig.savefig(path, dpi=150, bbox_inches='tight')
plt.show()
plt.close(fig)
print(f"Saved → {path}")

## 13 — Save Final Model

- `brain_tumor_vgg16.keras` — best checkpoint (already saved by ModelCheckpoint during training)
- `final_model.keras` — the model at the end of Phase 2 training

Both are saved in your `models/` directory.

In [ ]:
# Save the end-of-training model (ModelCheckpoint already saved the best one)
model.save(FINAL_MODEL_PATH)
print(f"✅ Final model saved → {FINAL_MODEL_PATH}")

# Confirm best model exists
if os.path.exists(BEST_MODEL_PATH):
    size_mb = os.path.getsize(BEST_MODEL_PATH) / 1e6
    print(f"✅ Best model exists → {BEST_MODEL_PATH}  ({size_mb:.1f} MB)")
else:
    print(f"⚠️  Best model not found at {BEST_MODEL_PATH}")

print("\n" + "═"*55)
print("  Training complete. Summary:")
print("═"*55)
best_acc = max(
    max(history1.history['val_accuracy']),
    max(history2.history['val_accuracy'])
)
print(f"  Best val accuracy   : {best_acc:.4f}")
print(f"  Phase 1 duration    : {phase1_time/60:.1f} min")
print(f"  Phase 2 duration    : {phase2_time/60:.1f} min")
print(f"  Best model saved at : {BEST_MODEL_PATH}")
print("═"*55)

## Training Complete ✅

Your model has been retrained and saved. The file structure is now:

```
models/
├── brain_tumor_vgg16.keras   ← best checkpoint (used by app.py)
└── final_model.keras         ← end of Phase 2

outputs/
├── plots/
│   ├── class_distribution.png
│   ├── sample_augmented.png
│   ├── history_phase1.png
│   ├── history_phase2.png
│   ├── history_combined.png
│   ├── confusion_matrix.png
│   ├── roc_curves.png
│   ├── gradcam_grid.png
│   └── prediction_card.png
└── reports/
    └── classification_report.txt
```

**Next steps:**
1. Run `streamlit run app.py` to launch the web interface
2. Upload any brain MRI image to get a prediction with Grad-CAM explanation